In [3]:
import pandas as pd 
import numpy as np
results=pd.read_csv('0_Datasets-results and geojson/svr_result.csv')

In [6]:
results=results.astype({"Potential": int, "Real": int, "county": str, 'Difference':int})
results=results[['county','Potential', 'Real', 'Difference']]
results

,county,Potential,Real,Difference
0,Appling,1769,1778,-9
1,Atkinson,563,825,-261
2,Bacon,1145,625,520
3,Baker,295,652,-356
4,Baldwin,14286,9140,5146
...,...,...,...,...
154,Whitfield,6947,10670,-3722
155,Wilcox,895,861,33
156,Wilkes,1496,2159,-662
157,Wilkinson,1234,2075,-840


In [7]:
key=pd.read_csv('0_Datasets-results and geojson/uszips.csv')[['zip','lat','lng','state_id','county_name']]
key=key[key.state_id=='GA']
def get_zip(county):
    return key[key.county_name==county].zip.iloc[0]
results['zip']=results.apply(lambda x: get_zip(x['county']),axis=1)

In [19]:
import requests
import geojsonio
import json
response=requests.get('https://raw.githubusercontent.com/python-visualization/folium/master/tests/us-counties.json')
geo_data=response.json()

#Filtering only in Georgia
geo=list()
for i in range(len(geo_data['features'])):
    if int(geo_data['features'][i]['id'])>=13001 and int(geo_data['features'][i]['id'])<=13321:
        geo.append(geo_data['features'][i])
geo_data['features']=geo

open('zipcode.geojson','w').write(
    json.dumps(geo_data,indent=4))

211444

In [22]:
bins

[-35405.0, -839.5, -107.0, 835.5, 40026.0]

In [36]:
# results.Difference=np.where(results.Difference>=0,0,results.Difference)
# results.Difference=-results.Difference

import folium
#Center the map at Times Square
#bins = [-35405.0,-20000.0,-10000.0,0.0,10000.0,20000.0,40026.0]

m = folium.Map(location = [33.788, -84.39],zoom_start=8)

folium.Choropleth(geo_data='0_Datasets-results and geojson/zipcode.geojson', data=results,
                 columns=['county','Difference'],
                 key_on='feature.properties.name',
                 fill_color='RdYlGn', 
                 fill_opacity=0.5, line_opacity=0.8,
                 bins=[-35405.0,-20000.0,-10000.0,0.0,10000.0,20000.0,40026.0],
                 legend_name='Difference Real Votes - Socio Economic Model ').add_to(m)
folium.LayerControl().add_to(m)
m

In [37]:
results

,county,Potential,Real,Difference,zip
0,Appling,1769,1778,-9,31513
1,Atkinson,563,825,-261,31624
2,Bacon,1145,625,520,31510
3,Baker,295,652,-356,39870
4,Baldwin,14286,9140,5146,31061
...,...,...,...,...,...
154,Whitfield,6947,10670,-3722,30710
155,Wilcox,895,861,33,31001
156,Wilkes,1496,2159,-662,30660
157,Wilkinson,1234,2075,-840,31003
